# NASA Mission Intelligence: end-to-end test run

**Author:** Dennis O'Higgins

This notebook runs the complete workflow on Google Colab: it installs the project, builds the ChromaDB collection with the embedding pipeline, inspects the collection, runs the batch RAGAS evaluation on `evaluation_dataset.txt`, and tests the Streamlit chat application with real questions.

The OpenAI API key is read from Colab Secrets (`OPENAI_API_KEY`; keys beginning with `voc-` are sent to the Udacity/Vocareum endpoint automatically, and an optional `OPENAI_BASE_URL` secret overrides the endpoint) and is never printed or saved.

## 1. Install the project and the NASA documents

In [1]:
!rm -rf /content/nasa /content/udc
!git clone -q https://github.com/dohigg1987/nasa-mission-rag-chat.git /content/nasa
!git clone -q --depth 1 https://github.com/udacity/cd13318-exercises-project.git /content/udc
%cd /content/nasa
!cp -r "/content/udc/Project-NASA-Mission-Intelligence-Starter/data_text" .
!pip -q install -r requirements.txt
!python fix_ragas_imports.py
!python -W ignore -c "import ragas, chromadb, openai, streamlit; print('ragas', ragas.__version__, '| chromadb', chromadb.__version__, '| openai', openai.__version__, '| streamlit', streamlit.__version__)"
!ls data_text/*

/content/nasa
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 572.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 74.1 MB/s eta 0:00

## 2. Load the OpenAI key from Colab Secrets

In [1]:
import os
from google.colab import userdata
os.chdir("/content/nasa")  # project folder created in step 1

key = None
for name in ("OPENAI_API_KEY", "OPENAI_API_Key", "OPENAI_KEY"):  # accept common spellings of the secret name
    try:
        key = userdata.get(name)
        break
    except Exception:
        continue
os.environ["OPENAI_API_KEY"] = key or ""
try:
    base_url = userdata.get("OPENAI_BASE_URL")
    if base_url:
        os.environ["OPENAI_BASE_URL"] = base_url
except Exception:
    pass  # a standard OpenAI key needs no base URL
os.environ["CHROMA_OPENAI_API_KEY"] = os.environ["OPENAI_API_KEY"]
print("API key loaded:", bool(os.environ.get("OPENAI_API_KEY")))
print("Custom base URL set:", bool(os.environ.get("OPENAI_BASE_URL")))

API key loaded: True
Custom base URL set: False


## 3. Build the ChromaDB collection (chunk size 500, overlap 100)

In [3]:
!python embedding_pipeline.py --data-path ./data_text --chroma-dir ./chroma_db_openai \
    --collection-name nasa_space_missions_text --chunk-size 500 --chunk-overlap 100 \
    --batch-size 100 --update-mode skip --test-query "What problem did Apollo 13 encounter?" 2>&1 \
    | grep -vE "Processing |chunks: "

2026-10-07 11:48:53,999 - INFO - Initializing ChromaDB Embedding Pipeline...
2026-10-07 11:48:54,321 - INFO - Using collection 'nasa_space_missions_text' in ./chroma_db_openai (0 chunks currently stored)
2026-10-07 11:48:54,322 - INFO - Starting text data processing with update mode: skip
2026-10-07 11:48:54,322 - INFO - Scanning directory: data_text/apollo11
2026-10-07 11:48:54,322 - INFO - Found 6 text files in apollo11
2026-10-07 11:48:54,322 - INFO - Scanning directory: data_text/apollo13
2026-10-07 11:48:54,323 - INFO - Found 3 text files in apollo13
2026-10-07 11:48:54,323 - INFO - Scanning directory: data_text/challenger
2026-10-07 11:48:54,323 - INFO - Found 3 text files in challenger
2026-10-07 11:48:54,323 - INFO - Total text files to process: 12
2026-10-07 11:48:54,323 - INFO - Files by mission:
2026-10-07 11:48:54,323 - INFO -   apollo_11: 6 files
2026-10-07 11:48:54,323 - INFO -   apollo_13: 3 files
2026-10-07 11:48:54,324 - INFO -   challenger: 3 files
2026-10-07 11:48:55

## 4. Inspect the collection (`--stats-only`)

In [4]:
!python embedding_pipeline.py --stats-only --chroma-dir ./chroma_db_openai \
    --collection-name nasa_space_missions_text 2>&1 | sed 's/^.* - INFO - //'

Initializing ChromaDB Embedding Pipeline...
Using collection 'nasa_space_missions_text' in ./chroma_db_openai (16512 chunks currently stored)
Collection Statistics:
collection_name: nasa_space_missions_text
collection_size (chunks): 16512
total_documents: 16512
missions: {'apollo_11': 8878, 'apollo_13': 6513, 'challenger': 1121}
data_types: {'textract_extracted': 10210, 'transcript': 6302}
document_categories: {'technical_report': 1000, 'public_affairs_officer': 6408, 'command_module': 880, 'flight_plan': 626, 'nasa_archive': 1313, 'technical': 5164, 'mission_audio': 1121}
file_types: {'text': 16512}
unique_source_files: 12


## 5. Re-run in `skip` mode: existing chunks are detected and not re-embedded

In [5]:
!python embedding_pipeline.py --data-path ./data_text --chroma-dir ./chroma_db_openai \
    --collection-name nasa_space_missions_text --chunk-size 500 --chunk-overlap 100 \
    --update-mode skip 2>&1 | grep -E "Documents added|Documents updated|Documents skipped|Total documents" | sed 's/^.* - INFO - //'

Documents added to collection: 0
Documents updated in collection: 0
Documents skipped (already exist): 16512
Total documents in collection: 16512


**Note on this run:** the first attempt at step 6 showed that ChromaDB 1.5.7 rejects question embeddings returned as numpy float32 values. `rag_client.py` was fixed to pass plain floats, the fix was pulled into the runtime, the Python session was restarted and step 2 was re-run before continuing from step 6. This is why the execution counts restart from step 2. The database built in steps 3 to 5 was kept.

## 6. Retrieval check: top-k with and without a mission filter

In [2]:
import rag_client
collection, ok, error = rag_client.initialize_rag_system("./chroma_db_openai", "nasa_space_missions_text")
print("Collection ready:", ok, error or "", "| chunks:", collection.count())
for mission in ["all", "apollo_13"]:
    res = rag_client.retrieve_documents(collection, "What problem did Apollo 13 encounter?", n_results=4, mission_filter=mission)
    print(f"\nmission_filter={mission}: {len(res['documents'][0])} results")
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print(f"  {dist:.3f}  {meta['mission']:<10} {meta['source'][:45]:<45} chunk {meta['chunk_index']}")
print()
print(rag_client.format_context(res["documents"][0][:2], res["metadatas"][0][:2])[:1500])

Collection ready: True  | chunks: 16512

mission_filter=all: 4 results
  0.382  apollo_13  AS13_PAO_textract_full_text                   chunk 869
  0.385  apollo_13  AS13_PAO_textract_full_text                   chunk 592
  0.388  apollo_13  AS13_PAO_textract_full_text                   chunk 2341
  0.390  apollo_13  AS13_PAO_textract_full_text                   chunk 3141

mission_filter=apollo_13: 4 results
  0.382  apollo_13  AS13_PAO_textract_full_text                   chunk 869
  0.385  apollo_13  AS13_PAO_textract_full_text                   chunk 592
  0.388  apollo_13  AS13_PAO_textract_full_text                   chunk 2341
  0.390  apollo_13  AS13_PAO_textract_full_text                   chunk 3141

NASA mission archive excerpts (most relevant first):

[Source 1] Mission: Apollo 13 | Document: AS13_PAO_textract_full_text | Category: Public Affairs Officer
command module. Isolating
the search tank leaves oxygen for entry if this should become
necessary. Also, if necessary, t

## 7. Batch evaluation on `evaluation_dataset.txt`

In [3]:
!python -W ignore batch_evaluation.py --dataset evaluation_dataset.txt --k 5 2>&1 | grep -vE "Evaluating|it/s\]"

Loaded 7 questions from evaluation_dataset.txt

[1/7] (overview) What was the main objective of the Apollo 11 mission?
Answer: The main objective of the Apollo 11 mission was to perform a manned lunar landing and return safely to Earth [Source 2].
Retrieved chunks: 5
Scores: response_relevancy=1.000, faithfulness=0.500, context_precision=1.000, bleu_score=0.314, rouge_score=0.493

[2/7] (emergency) What problem did Apollo 13 encounter, and what were its consequences?
Answer: Apollo 13 encountered an oxygen tank explosion that caused a loss of service module capability. This led to the mission being aborted, the lunar landing being canceled, and the crew having to use the lunar module as a lifeboat to return safely to Earth [Source 1]. The explosion occurred at approximately 55 hours, 55 minutes into the mission [Source 1].
Retrieved chunks: 5
Scores: response_relevancy=0.600, faithfulness=0.000, context_precision=0.367, bleu_score=0.050, rouge_score=0.458

[3/7] (disaster analysis) Wha

## 8. Streamlit chat application test (real questions, real scores)

In [4]:
from streamlit.testing.v1 import AppTest

at = AppTest.from_file("chat.py", default_timeout=300).run()
print("Sidebar controls:", [s.label for s in at.selectbox], [s.label for s in at.slider])
at.selectbox[2].set_value("apollo_13").run()   # mission focus

for question in ["Who were the Apollo 13 crew members?",
                 "What did they do after the explosion?"]:  # follow-up uses conversation history
    at.chat_input[0].set_value(question).run()
    print("\nQ:", question)
    print("A:", [m.markdown[0].value for m in at.chat_message if m.markdown][-1])
    print("Scores:", [(m.label, m.value) for m in at.metric])
print("\nExceptions:", list(at.exception))

2026-10-07 12:03:20.221 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Sidebar controls: ['Select Document Collection', 'OpenAI Model', 'Mission focus'] ['Documents to retrieve']

Q: Who were the Apollo 13 crew members?
A: The Apollo 13 crew members were Jim Lovell, Jack Swigert, and Fred Haise. They were all reported to be feeling fine and were about to begin their scheduled 10-hour rest period [Source 1].
Scores: [('Response Relevancy', '0.751'), ('Faithfulness', '0.600'), ('Context Precision', '1.000')]

Q: What did they do after the explosion?
A: After the explosion on Apollo 13, the crew assessed the damage caused by the explosion. They observed that there was debris hanging out the side near the S-band antenna [Source 2]. They also discussed the extensive damage and tried to locate it [Source 3]. The crew then worked on stabilizing the spacecraft and implementing procedures to conserve power and resources for their return to Earth.
Scores: [('Response Relevancy', '0.676'), ('Faithfulness', '0.400'), ('Context Precision', '1.000')]

Exceptions: []


## 9. Package the results

In [5]:
!zip -q -j /content/nasa_results.zip evaluation_results.json evaluation_results.csv chroma_embedding_text_only.log
!ls -la /content/nasa_results.zip evaluation_results.*

-rw-r--r-- 1 root root 8444 Oct  7 12:04 /content/nasa_results.zip
-rw-r--r-- 1 root root 5171 Oct  7 12:03 evaluation_results.csv
-rw-r--r-- 1 root root 7545 Oct  7 12:03 evaluation_results.json
